# Audit 3 — Row position effects in the OP3 benchmark targets (NeurIPS 2024, peer-reviewed)

**Benchmark:** OP3 (Open Problems perturbation prediction; NeurIPS 2024 Datasets & Benchmarks). Targets are differential-expression statistics per compound x cell type against DMSO, from limma with the design `~ 0 + sm_cell_type + plate_name`, i.e. **no row or column terms**.

**Layout:** each plate has DMSO (column 3), belinostat (column 1) and dabrafenib (column 2) in **every row A-H**; compounds occupy columns 4-12, one well each. Because the three control columns span all rows, **row effects can be estimated from control wells alone**, independently of the compounds.

**Question:** do compound responses contain the row effect of the row they sit in? If so, benchmark targets mix compound biology with plate position, and a model can be rewarded for predicting position.

**Tests:**
1. **Layout:** does each compound occupy the same well on every plate? (If yes, position and compound are confounded in the targets.)
2. **Row effects from controls:** for each plate and cell type, the deviation of each row from the column mean, averaged over the three control compounds. Are row effects reproducible across plates (systematic) or random?
3. **Contamination:** correlation between a compound's response (vs the plate's DMSO mean) and the row effect of its own row, compared with the row effects of the other rows. Row effects come from different wells than the compound, so a positive excess can only be shared position signal.

Uses the pseudobulk cache from `op3_analysis.ipynb` (`data/op3/cache/pb_well.npz`).

In [1]:
import os
import numpy as np, pandas as pd
import anndata as ad
from scipy.stats import rankdata
from itertools import combinations
import matplotlib.pyplot as plt, seaborn as sns

OP3_FILE = "data/op3/sc_counts_reannotated_with_counts.h5ad"
CACHE = "data/op3/cache/pb_well.npz"
RESULTS_DIR = "results/op3_benchmark_audit"
os.makedirs(RESULTS_DIR, exist_ok=True)
CONTROLS = ["Dimethyl Sulfoxide", "Belinostat", "Dabrafenib"]
DMSO = "Dimethyl Sulfoxide"
CELL_TYPES = ["T cells", "B cells", "Myeloid cells", "NK cells"]
ROWS = list("ABCDEFGH")
MIN_CELLS = 30
N_GENES = 2000
N_PERM = 1000
SEED = 0

z = np.load(CACHE, allow_pickle=True)
SUMS = z["sums"]; PB = pd.DataFrame(z["meta"].tolist(), columns=["plate", "well", "cell_type", "half", "n_cells"])
obs = ad.read_h5ad(OP3_FILE, backed="r").obs
WELLS = (obs.groupby(["plate_name", "well"], observed=True)
         .agg(row=("row", "first"), col=("col", "first"), compound=("sm_name", "first")).reset_index()
         .rename(columns={"plate_name": "plate"}))
for c in ["plate", "well", "row", "compound"]: WELLS[c] = WELLS[c].astype(str)
WELLS["col"] = WELLS["col"].astype(int)

def lognorm(x): return np.log1p(x / max(x.sum(), 1) * 1e4)
GIDX = np.argsort(lognorm(SUMS.sum(0)))[::-1][:N_GENES]
FULL = PB.groupby(["plate", "well", "cell_type"]).apply(lambda g: (g.index.to_numpy(), g.n_cells.sum())).to_dict()
def prof(plate, well, ct):
    v = FULL.get((plate, well, ct))
    if v is None or v[1] < MIN_CELLS: return None
    return lognorm(SUMS[v[0]].astype(np.float64).sum(0))[GIDX]
PLATES = sorted(WELLS.plate.unique())

## 1. Layout: is each compound in the same well on every plate?

In [2]:
comp = WELLS[~WELLS.compound.isin(CONTROLS)]
pos = comp.groupby("compound").apply(lambda g: g[["row", "col"]].drop_duplicates().shape[0])
plates_per = comp.groupby("compound").plate.nunique()
LAYOUT = pd.DataFrame({"n_plates": plates_per, "n_distinct_positions": pos})
print(LAYOUT.n_plates.value_counts().rename("compounds by number of plates").to_dict())
print(f"compounds always in the same well: {(LAYOUT.n_distinct_positions == 1).mean():.0%}")
LAYOUT.to_csv(f"{RESULTS_DIR}/compound_positions.csv")

{3: 141}
compounds always in the same well: 100%


## 2. Row effects estimated from the control columns

In [3]:
ROWEFF = {}          # (plate, cell_type) -> {row: effect vector}
for plate in PLATES:
    W = WELLS[WELLS.plate == plate]
    for ct in CELL_TYPES:
        devs = {r: [] for r in ROWS}
        for k in CONTROLS:
            wk = W[W.compound == k]
            P = {r.row: prof(plate, r.well, ct) for r in wk.itertuples()}
            P = {r: v for r, v in P.items() if v is not None}
            if len(P) < 6: continue
            m = np.mean(list(P.values()), 0)
            for r, v in P.items(): devs[r].append(v - m)
        eff = {r: np.mean(v, 0) for r, v in devs.items() if len(v) >= 2}
        if len(eff) >= 6: ROWEFF[(plate, ct)] = eff

def corr(a, b):
    a = a - a.mean(); b = b - b.mean(); return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))
rows = []
for ct in CELL_TYPES:
    keys = [k for k in ROWEFF if k[1] == ct]
    for (p1, _), (p2, _) in combinations(keys, 2):
        E1, E2 = ROWEFF[(p1, ct)], ROWEFF[(p2, ct)]
        common = [r for r in ROWS if r in E1 and r in E2]
        same = np.mean([corr(E1[r], E2[r]) for r in common])
        diff = np.mean([corr(E1[r], E2[s]) for r in common for s in common if s != r])
        rows.append(dict(cell_type=ct, plates=f"{p1} vs {p2}", same_row=same, different_rows=diff))
ROWCONS = pd.DataFrame(rows)
ROWCONS.to_csv(f"{RESULTS_DIR}/row_effect_consistency.csv", index=False)
print("Row effects: correlation of the same row across plates vs different rows")
ROWCONS.groupby("cell_type")[["same_row", "different_rows"]].median().round(3)

Row effects: correlation of the same row across plates vs different rows


,same_row,different_rows
cell_type,,
B cells,0.025,-0.003
Myeloid cells,0.032,-0.001
NK cells,0.036,-0.006
T cells,0.037,-0.006


## 3. Do compound responses contain the row effect of their own row?

In [4]:
rng = np.random.default_rng(SEED)
rows = []
for (plate, ct), eff in ROWEFF.items():
    W = WELLS[WELLS.plate == plate]
    dm = [prof(plate, w, ct) for w in W[W.compound == DMSO].well]
    dm = [v for v in dm if v is not None]
    if len(dm) < 4: continue
    ref = np.mean(dm, 0)
    X = []
    for r in W[~W.compound.isin(CONTROLS)].itertuples():
        v = prof(plate, r.well, ct)
        if v is not None and r.row in eff: X.append((r.row, v - ref))
    if len(X) < 20: continue
    matched = np.mean([corr(x, eff[r]) for r, x in X])
    others = np.mean([corr(x, eff[s]) for r, x in X for s in eff if s != r])
    # permutation: shuffle which row effect belongs to which row
    labels = list(eff)
    null = []
    for _ in range(N_PERM):
        perm = dict(zip(labels, rng.permutation(labels)))
        null.append(np.mean([corr(x, eff[perm[r]]) for r, x in X]))
    rows.append(dict(plate=plate, cell_type=ct, n_wells=len(X), matched=matched, other_rows=others,
                     excess=matched - others, p=(1 + np.sum(np.array(null) >= matched)) / (1 + N_PERM)))
CONTAM = pd.DataFrame(rows)
CONTAM.to_csv(f"{RESULTS_DIR}/row_contamination.csv", index=False)
CONTAM_SUMMARY = CONTAM.groupby("cell_type")[["matched", "other_rows", "excess"]].median().round(3)
CONTAM_SUMMARY["plates_p_below_005"] = CONTAM.groupby("cell_type").p.apply(lambda p: f"{(p < 0.05).sum()}/{len(p)}")
CONTAM_SUMMARY.to_csv(f"{RESULTS_DIR}/row_contamination_summary.csv")
CONTAM_SUMMARY

,matched,other_rows,excess,plates_p_below_005
cell_type,,,,
B cells,0.089,-0.007,0.094,6/6
Myeloid cells,0.083,-0.007,0.095,4/6
NK cells,0.079,-0.004,0.086,3/3
T cells,0.156,-0.009,0.156,6/6


## How to read
- **Layout:** if compounds always sit in the same well, any position effect is inseparable from the compound effect in the benchmark targets.
- **Row-effect consistency:** `same_row` clearly above `different_rows` means row effects are systematic across plates, i.e. a property of the plate layout or handling.
- **Contamination:** `matched` above `other_rows` (permutation p < 0.05) means compound responses carry the row effect of their row, estimated from completely separate control wells. The targets then contain position signal that a model could learn and be rewarded for.